# Adversarial Patch Experiments — YOLOv8 / YOLO11 / YOLO26

**Capstone: Person-Vanishing Adversarial Patch Attack against Ultralytics YOLO**

Trains patches directly against each model on a **common 14-image manifest** (images where all three models detect ≥1 person), then runs cross-version transfer tests.

**Before running:** `Runtime > Change runtime type > T4 GPU`

## 1. Setup

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
REPO_URL = 'https://github.com/Cmaddock99/Patch_Yolo.git'
REPO_DIR = '/content/Adversarial_Patch'
!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Mount Google Drive and copy yolo26n.pt (not tracked in git — *.pt is gitignored)
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
src = '/content/drive/MyDrive/yolo26n.pt'
dst = '/content/Adversarial_Patch/yolo26n.pt'
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'Copied yolo26n.pt ({os.path.getsize(dst):,} bytes)')
else:
    raise FileNotFoundError(f'yolo26n.pt not found on Drive at {src} — upload it first')


In [ ]:
!pip install ultralytics opencv-python-headless tqdm -q

## 2. Manifest — Common 14-Image Subset

Finds images detected as 'person' by **all three** models simultaneously. This ensures fair cross-version comparisons on a fixed evaluation set.

In [ ]:
!python experiments/build_intersection_manifest.py

## 3. Forward Contract Diagnostic — YOLO26

Confirms the YOLO26 inner model output shape and that `one2many['scores']` has a `grad_fn` (gradient flows through it). Run once before training.

In [ ]:
import torch
from ultralytics import YOLO

yolo26 = YOLO('yolo26n.pt')
inner26 = yolo26.model.eval()
dummy = torch.zeros(1, 3, 640, 640)
with torch.enable_grad():
    out = inner26(dummy)
print('type(out):', type(out))
if isinstance(out, (list, tuple)):
    print('len(out):', len(out))
    print('out[0].shape:', out[0].shape)
    if isinstance(out[1], dict):
        for k, v in out[1].items():
            print(f"preds['{k}'] keys:", list(v.keys()))
            for kk, vv in v.items():
                if isinstance(vv, torch.Tensor):
                    print(f'  [{kk}] shape={vv.shape}, grad_fn={vv.grad_fn}')
del yolo26, inner26  # free VRAM before training

## 4. Direct Training — YOLOv8n

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolov8n \
    --manifest data/manifests/common_14.txt \
    --seed 42 \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --nps-weight 0.01 \
    --block-erase-prob 0.5 \
    --cutout-prob 0.3 \
    --rot-max 15.0 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
from IPython.display import Image as IPImage, display
import json
display(IPImage('outputs/yolov8n_patch_v2/patches/patch.png', width=200))
with open('outputs/yolov8n_patch_v2/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 5. Direct Training — YOLO11n

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolo11n \
    --manifest data/manifests/common_14.txt \
    --seed 42 \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --nps-weight 0.01 \
    --block-erase-prob 0.5 \
    --cutout-prob 0.3 \
    --rot-max 15.0 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
import json
with open('outputs/yolo11n_patch_v2/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 6. Direct Training — YOLO26n

Root cause of v1 failure: `end2end=True` returns `(B, 300, 6)` as `out[0]`. Fixed to use `preds['one2many']['scores']` `(B, 80, 8400)` which retains gradient flow.

**LR note:** Using `--lr 0.001` (10× lower than v8/v11). The one2many gradient landscape has a different scale than v8/v11 class scores — a lower learning rate prevents the patch from diverging early.

In [ ]:
import ultralytics
print('Ultralytics version:', ultralytics.__version__)
from ultralytics import YOLO
try:
    m = YOLO('yolo26n.pt')
    print('yolo26n loaded OK')
except Exception as e:
    print('Error:', e)

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolo26n \
    --manifest data/manifests/common_14.txt \
    --seed 42 \
    --epochs 1000 \
    --lr 0.001 \
    --tv-weight 0.05 \
    --nps-weight 0.01 \
    --block-erase-prob 0.5 \
    --cutout-prob 0.3 \
    --rot-max 15.0 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5

In [ ]:
import json
with open('outputs/yolo26n_patch_v2/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 7. Cross-Version Transfer Tests

Patch trained on YOLOv8n — evaluated against YOLO11n and YOLO26n. Uses the same common manifest for a fair comparison.

Output dirs: `yolo11n_from_yolov8n_patch_v2_transfer/`, etc.

In [ ]:
# v8 patch → v11
!python experiments/ultralytics_patch.py \
    --model yolo11n --eval-only \
    --load-patch outputs/yolov8n_patch_v2/patches/patch.png \
    --manifest data/manifests/common_14.txt \
    --display 3

In [ ]:
# v8 patch → v26
!python experiments/ultralytics_patch.py \
    --model yolo26n --eval-only \
    --load-patch outputs/yolov8n_patch_v2/patches/patch.png \
    --manifest data/manifests/common_14.txt \
    --display 3

In [ ]:
# v11 patch → v26
!python experiments/ultralytics_patch.py \
    --model yolo26n --eval-only \
    --load-patch outputs/yolo11n_patch_v2/patches/patch.png \
    --manifest data/manifests/common_14.txt \
    --display 3

## 10. Multi-Model Joint Training — YOLOv8n + YOLO11n

Trains a **single patch** that minimises detection loss on both YOLOv8n and YOLO11n simultaneously (losses averaged each step). Literature basis: DOEPatch (Tan 2024), T-SEA (Huang 2022) ensemble approach.

Hypothesis: a jointly-trained patch should transfer better to YOLO26n than a single-model patch because it has seen a broader gradient landscape during training.

In [ ]:
!python experiments/ultralytics_patch.py \
    --model yolov8n \
    --co-model yolo11n \
    --manifest data/manifests/common_14.txt \
    --seed 42 \
    --epochs 1000 \
    --lr 0.01 \
    --tv-weight 0.05 \
    --nps-weight 0.01 \
    --block-erase-prob 0.5 \
    --cutout-prob 0.3 \
    --rot-max 15.0 \
    --patch-size 100 \
    --batch-size 8 \
    --display 5
# Output: outputs/yolov8n+yolo11n_joint_patch_v2/

In [ ]:
from IPython.display import Image as IPImage, display
import json
display(IPImage('outputs/yolov8n+yolo11n_joint_patch_v2/patches/patch.png', width=200))
with open('outputs/yolov8n+yolo11n_joint_patch_v2/results.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 11. Cross-Version Transfer — Joint Patch → All Models

Evaluate the jointly-trained patch (v8n+v11n) on each target model. Compare transfer rates against the single-model v8n baseline from Section 7.

In [ ]:
# Joint patch → v8n (self-eval on primary model)
!python experiments/ultralytics_patch.py \
    --model yolov8n --eval-only \
    --load-patch outputs/yolov8n+yolo11n_joint_patch_v2/patches/patch.png \
    --manifest data/manifests/common_14.txt \
    --display 3

In [ ]:
# Joint patch → v11n (co-model self-eval)
!python experiments/ultralytics_patch.py \
    --model yolo11n --eval-only \
    --load-patch outputs/yolov8n+yolo11n_joint_patch_v2/patches/patch.png \
    --manifest data/manifests/common_14.txt \
    --display 3

In [ ]:
# Joint patch → v26n (key transfer test)
!python experiments/ultralytics_patch.py \
    --model yolo26n --eval-only \
    --load-patch outputs/yolov8n+yolo11n_joint_patch_v2/patches/patch.png \
    --manifest data/manifests/common_14.txt \
    --display 3

## 8. Results Summary

In [ ]:
import json, os
print(f"{'Experiment':<55} {'Clean':>6} {'Patched':>8} {'Suppression':>12}")
print('-' * 83)
for run_dir in sorted(os.listdir('outputs')):
    rp = f'outputs/{run_dir}/results.json'
    if os.path.exists(rp):
        with open(rp) as f:
            r = json.load(f)
        tag = ' [TRANSFER]' if 'transfer' in run_dir else ' [DIRECT]  '
        label = run_dir + tag
        print(f"{label:<55} {r.get('clean_person_detections','-'):>6} "
              f"{r.get('patched_person_detections','-'):>8} "
              f"{str(r.get('detection_suppression_pct','-'))+'%':>12}")

## 9. Push Results to GitHub

In [ ]:
from google.colab import userdata
import subprocess

TOKEN = userdata.get('GITHUB_PAT')  # set this in Colab Secrets (key icon in left sidebar)
REPO  = 'Cmaddock99/Patch_Yolo'

!git config user.email 'cmaddock99@gmail.com'
!git config user.name 'Cmaddock99'
!git add outputs/ experiments/ data/manifests/
!git commit -m 'Add v2 results: robustness improvements (DePatch+T-SEA+EoT) + joint training'
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{TOKEN}@github.com/{REPO}.git'], check=True)
!git push
